In [1]:
!pip install -q transformers datasets evaluate seqeval accelerate torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00


In [2]:
import datasets
import transformers

In [3]:
from tqdm import tqdm
for i in tqdm(range(100),disable=True):
    pass

In [4]:
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
    pipeline,
)

In [5]:
dataset=load_dataset("lhoestq/conll2003")
dataset

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/281k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/259k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [6]:
print(dataset["train"].features)

{'id': Value('string'), 'tokens': List(Value('string')), 'pos_tags': List(Value('int64')), 'chunk_tags': List(Value('int64')), 'ner_tags': List(Value('int64'))}


In [7]:
chunk_labels = [
    "O", "B-ADJP", "I-ADJP", "B-ADVP", "I-ADVP", "B-CONJP", "I-CONJP",
    "B-INTJ", "I-INTJ", "B-LST", "I-LST", "B-NP", "I-NP", "B-PP", "I-PP",
    "B-PRT", "I-PRT", "B-SBAR", "I-SBAR", "B-UCP", "I-UCP", "B-VP", "I-VP"
]

pos_labels = [
    "``", "''", "#", "$", "''", "(", ")", ",", ".", ":", "CC", "CD",
    "DT", "EX", "FW", "IN", "JJ", "JJR", "JJS", "LS", "MD", "NN", "NNS",
    "NNP", "NNPS", "PDT", "POS", "PRP", "PRPS", "RB", "RBR", "RBS", "RP",
    "SYM", "TO", "UH", "VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "WDT",
    "WP", "WP$", "WRB"
]
print(len(pos_labels))
print(len(chunk_labels))

46
23


In [8]:
id2label = {i: label for i, label in enumerate(chunk_labels)}
label2id = {label: i for i, label in enumerate(chunk_labels)}
print("id2label:", id2label)
print("label2id:", label2id)

id2label: {0: 'O', 1: 'B-ADJP', 2: 'I-ADJP', 3: 'B-ADVP', 4: 'I-ADVP', 5: 'B-CONJP', 6: 'I-CONJP', 7: 'B-INTJ', 8: 'I-INTJ', 9: 'B-LST', 10: 'I-LST', 11: 'B-NP', 12: 'I-NP', 13: 'B-PP', 14: 'I-PP', 15: 'B-PRT', 16: 'I-PRT', 17: 'B-SBAR', 18: 'I-SBAR', 19: 'B-UCP', 20: 'I-UCP', 21: 'B-VP', 22: 'I-VP'}
label2id: {'O': 0, 'B-ADJP': 1, 'I-ADJP': 2, 'B-ADVP': 3, 'I-ADVP': 4, 'B-CONJP': 5, 'I-CONJP': 6, 'B-INTJ': 7, 'I-INTJ': 8, 'B-LST': 9, 'I-LST': 10, 'B-NP': 11, 'I-NP': 12, 'B-PP': 13, 'I-PP': 14, 'B-PRT': 15, 'I-PRT': 16, 'B-SBAR': 17, 'I-SBAR': 18, 'B-UCP': 19, 'I-UCP': 20, 'B-VP': 21, 'I-VP': 22}


In [9]:
model_checkpoint = "distilbert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(chunk_labels),
    id2label=id2label,
    label2id=label2id,
)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/263M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
print("Model checkpoint:", model_checkpoint)
print("num_labels:", model.config.num_labels)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

Model checkpoint: distilbert-base-cased
num_labels: 23
id2label: {0: 'O', 1: 'B-ADJP', 2: 'I-ADJP', 3: 'B-ADVP', 4: 'I-ADVP', 5: 'B-CONJP', 6: 'I-CONJP', 7: 'B-INTJ', 8: 'I-INTJ', 9: 'B-LST', 10: 'I-LST', 11: 'B-NP', 12: 'I-NP', 13: 'B-PP', 14: 'I-PP', 15: 'B-PRT', 16: 'I-PRT', 17: 'B-SBAR', 18: 'I-SBAR', 19: 'B-UCP', 20: 'I-UCP', 21: 'B-VP', 22: 'I-VP'}
label2id: {'O': 0, 'B-ADJP': 1, 'I-ADJP': 2, 'B-ADVP': 3, 'I-ADVP': 4, 'B-CONJP': 5, 'I-CONJP': 6, 'B-INTJ': 7, 'I-INTJ': 8, 'B-LST': 9, 'I-LST': 10, 'B-NP': 11, 'I-NP': 12, 'B-PP': 13, 'I-PP': 14, 'B-PRT': 15, 'I-PRT': 16, 'B-SBAR': 17, 'I-SBAR': 18, 'B-UCP': 19, 'I-UCP': 20, 'B-VP': 21, 'I-VP': 22}


In [11]:
sample_tokens = dataset["train"][0]["tokens"]
sample_chunk_tags = dataset["train"][0]["chunk_tags"]
print("Tokens:", sample_tokens)
print("Chunk Tag IDs:", sample_chunk_tags)
print("Chunk Tag Names:", [chunk_labels[i] for i in sample_chunk_tags])

Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
Chunk Tag IDs: [11, 21, 11, 12, 21, 22, 11, 12, 0]
Chunk Tag Names: ['B-NP', 'B-VP', 'B-NP', 'I-NP', 'B-VP', 'I-VP', 'B-NP', 'I-NP', 'O']


In [12]:
encoded_sample = tokenizer(sample_tokens, is_split_into_words=True, truncation=True)
print("Input IDs:", encoded_sample["input_ids"])
print("Attention Mask:", encoded_sample["attention_mask"])
print("Tokens:", tokenizer.convert_ids_to_tokens(encoded_sample["input_ids"]))
print("Word IDs:", encoded_sample.word_ids())

Input IDs: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
Attention Mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Tokens: ['[CLS]', 'EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'la', '##mb', '.', '[SEP]']
Word IDs: [None, 0, 1, 2, 3, 4, 5, 6, 7, 7, 8, None]


In [13]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=128,
    )
    labels = []
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [14]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)
tokenized_datasets

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [15]:
print("input_ids:", tokenized_datasets["train"][0]["input_ids"][:30])
print("attention_mask:", tokenized_datasets["train"][0]["attention_mask"][:30])
print("labels:", tokenized_datasets["train"][0]["labels"][:30])

input_ids: [101, 7270, 22961, 1528, 1840, 1106, 21423, 1418, 2495, 12913, 119, 102]
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
labels: [-100, 11, 21, 11, 12, 21, 22, 11, 12, -100, 0, -100]


In [16]:
tokens = tokenizer.convert_ids_to_tokens(tokenized_datasets["train"][0]["input_ids"])
labels = tokenized_datasets["train"][0]["labels"]
for tok, lab in zip(tokens[:40], labels[:40]):
    if lab == -100:
        print(f"{tok:15} -> -100")
    else:
        print(f"{tok:15} -> {id2label[lab]}")

[CLS]           -> -100
EU              -> B-NP
rejects         -> B-VP
German          -> B-NP
call            -> I-NP
to              -> B-VP
boycott         -> I-VP
British         -> B-NP
la              -> I-NP
##mb            -> -100
.               -> O
[SEP]           -> -100


In [17]:
seqeval = evaluate.load("seqeval")

In [18]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [chunk_labels[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [chunk_labels[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [19]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [20]:
training_args = TrainingArguments(
    output_dir="./distilbert-chunking",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

In [22]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.318512,0.182670,0.913163,0.903226,0.908167,0.951815
2,0.147422,0.162816,0.919832,0.911038,0.915414,0.955712
3,0.115652,0.160273,0.921552,0.915857,0.918696,0.957329


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=2634, training_loss=0.1938618462286669, metrics={'train_runtime': 262.1602, 'train_samples_per_second': 160.677, 'train_steps_per_second': 10.047, 'total_flos': 524901061281240.0, 'train_loss': 0.1938618462286669, 'epoch': 3.0})

In [23]:
validation_results = trainer.evaluate()
validation_results

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


{'eval_loss': 0.1602725386619568,
 'eval_precision': 0.9215517578506902,
 'eval_recall': 0.9158569762922658,
 'eval_f1': 0.9186955419972319,
 'eval_accuracy': 0.9573290727354208,
 'eval_runtime': 7.175,
 'eval_samples_per_second': 452.961,
 'eval_steps_per_second': 28.432,
 'epoch': 3.0}

In [24]:
test_results = trainer.evaluate(tokenized_datasets["test"])
test_results

{'eval_loss': 0.18612520396709442,
 'eval_precision': 0.9115877462912781,
 'eval_recall': 0.9015261627906976,
 'eval_f1': 0.9065290369727557,
 'eval_accuracy': 0.9538962018226081,
 'eval_runtime': 6.4087,
 'eval_samples_per_second': 538.803,
 'eval_steps_per_second': 33.704,
 'epoch': 3.0}

In [25]:
print("Precision:", round(test_results["eval_precision"], 4))
print("Recall:", round(test_results["eval_recall"], 4))
print("F1 Score:", round(test_results["eval_f1"], 4))
print("Accuracy:", round(test_results["eval_accuracy"], 4))

Precision: 0.9116
Recall: 0.9015
F1 Score: 0.9065
Accuracy: 0.9539


In [26]:
trainer.save_model("./distilbert-chunking-final")
tokenizer.save_pretrained("./distilbert-chunking-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./distilbert-chunking-final/tokenizer_config.json',
 './distilbert-chunking-final/tokenizer.json')

In [27]:
chunking_pipeline = pipeline(
    "token-classification",
    model="./distilbert-chunking-final",
    tokenizer="./distilbert-chunking-final",
    aggregation_strategy="simple",
)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

In [28]:
text = "The quick brown fox jumps over the lazy dog"
predictions = chunking_pipeline(text)
print("Input:", text)
for entity in predictions:
    print(f"Word/Phrase: {entity['word']} | Tag: {entity['entity_group']} | Score: {entity['score']:.4f}")

Input: The quick brown fox jumps over the lazy dog
Word/Phrase: The quick brown fox | Tag: NP | Score: 0.9967
Word/Phrase: jumps | Tag: VP | Score: 0.9457
Word/Phrase: over | Tag: PP | Score: 0.9172
Word/Phrase: the lazy dog | Tag: NP | Score: 0.9980


In [29]:
pos_id2label = {i: label for i, label in enumerate(pos_labels)}
pos_label2id = {label: i for i, label in enumerate(pos_labels)}
print("POS id2label:", pos_id2label)

POS id2label: {0: '``', 1: "''", 2: '#', 3: '$', 4: "''", 5: '(', 6: ')', 7: ',', 8: '.', 9: ':', 10: 'CC', 11: 'CD', 12: 'DT', 13: 'EX', 14: 'FW', 15: 'IN', 16: 'JJ', 17: 'JJR', 18: 'JJS', 19: 'LS', 20: 'MD', 21: 'NN', 22: 'NNS', 23: 'NNP', 24: 'NNPS', 25: 'PDT', 26: 'POS', 27: 'PRP', 28: 'PRPS', 29: 'RB', 30: 'RBR', 31: 'RBS', 32: 'RP', 33: 'SYM', 34: 'TO', 35: 'UH', 36: 'VB', 37: 'VBD', 38: 'VBG', 39: 'VBN', 40: 'VBP', 41: 'VBZ', 42: 'WDT', 43: 'WP', 44: 'WP$', 45: 'WRB'}


In [30]:
sample_pos_tags = dataset["train"][0]["pos_tags"]
print("Tokens:", sample_tokens)
print("POS Tag IDs:", sample_pos_tags)
print("POS Tag Names:", [pos_labels[i] for i in sample_pos_tags])

Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
POS Tag IDs: [22, 42, 16, 21, 35, 37, 16, 21, 7]
POS Tag Names: ['NNS', 'WDT', 'JJ', 'NN', 'UH', 'VBD', 'JJ', 'NN', ',']


In [31]:
for token, pos_tag, chunk_tag in zip(
    dataset["train"][0]["tokens"],
    dataset["train"][0]["pos_tags"],
    dataset["train"][0]["chunk_tags"]
):
    print(
        f"Token: {token:15} | POS: {pos_labels[pos_tag]:10} | Chunk: {chunk_labels[chunk_tag]}"
    )

Token: EU              | POS: NNS        | Chunk: B-NP
Token: rejects         | POS: WDT        | Chunk: B-VP
Token: German          | POS: JJ         | Chunk: B-NP
Token: call            | POS: NN         | Chunk: I-NP
Token: to              | POS: UH         | Chunk: B-VP
Token: boycott         | POS: VBD        | Chunk: I-VP
Token: British         | POS: JJ         | Chunk: B-NP
Token: lamb            | POS: NN         | Chunk: I-NP
Token: .               | POS: ,          | Chunk: O


POS Tagging (Word-Level Tagging): It involves assigning grammatical labels to each individual word. (Example: John – Noun, works – Verb). This task is less complex for machines since it requires only a small context window to assign POS tags.

Chunking (Phrase-level grouping): Here, words that are related are grouped together into phrases (e.g., “The quick brown fox” = Noun Phrase). This task has a medium level of difficulty since, apart from recognizing word types, the model should also recognize phrases’ boundaries using the BIO tag set.

In POS tagging, one deals with high detail: "What grammatical function does this particular word play?" This approach focuses on individual tokens, where 'Google' becomes a Proper Noun and 'in' a Preposition.

In chunking, one deals with structure: "How do words combine into meaning units?" This task focuses on spans of text: it sees "Google in California" as being part of phrase construction with the help of BIO (beginning, inside, outside) tags for marking up phrase boundaries.

Challenges

The major difficulty that comes up in utilizing BERT-type models for token labeling is related to the mismatch between the word in question and the tokens generated through the tokenization process carried out by the model. The model uses sub-word tokenization, such as WordPiece. For instance, "tokenization" can be tokenized into "token," "##iza," and "##tion." If there is only one label linked to the whole word, then the problem is based on the lack of consistency. The approach that was used for solving this issue is masking, whereby the real label is given to the first token, "token," while the other two tokens are labeled "-100." The label is used as an ignore index in PyTorch's CrossEntropyLoss.



Observations and insights

The use of DistilBERT resulted in huge speedups during training without losing accuracy. The performance of the model was measured by using seqeval, a metric that takes into account the entire sequence as opposed to looking at each token individually. This gives us the Precision, Recall, and F1 Score. The model could easily grasp POS tags because of their locality, but took longer to recognize the correct phrase boundaries required for Chunking.